<a href="https://colab.research.google.com/github/yoshiagent/automaticControl-colab/blob/main/notebooks/Simulink_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# Simulink 模擬入門：轉移函數、狀態空間、積分器、線性、頻率響應

對照網頁：[自動控制｜模擬](https://yoshiagent.github.io/automaticControl/SIM.html)、
[第五週：系統模擬－時域頻域](https://yoshiagent.github.io/automaticControl/WK-5.html)、
[系統（線性）](https://yoshiagent.github.io/automaticControl/MM-System.html)

原課程用 Simulink 模型（`.mdl`）示範。這本筆記本把每個模型的方塊與參數（直接從 `.mdl` 檔讀出）改寫成 Python：

| Simulink 方塊 | Python 寫法 |
|---|---|
| Integrator（積分器） | 一個狀態變數：`x' = 輸入` |
| Gain、Sum | 乘法、加減（Sum 的 `"+-"` 符號照抄） |
| Transfer Fcn、State-Space | `TF(num, den)`（轉成狀態空間）、`A @ x + B * u` |
| Step、Ramp、Sine Wave | `step(t)`、`ramp(t)`、`np.sin(t)`（預設值同 Simulink：Step 在 $t=1$ 由 0 跳到 1） |
| Scope、To Workspace | 用 plotly 畫圖、存成陣列 |
| Solver ode45 | `scipy.integrate.solve_ivp`（RK45，同一族方法） |

| 原檔 | 內容 | 本筆記本 |
|---|---|---|
| `TfSim.mdl`、`sssim.mdl`、`IntgratorM.mdl` | 同一個系統 $\ddot x+5\dot x+6x=10\sin t$ 的三種建模方式 | 第 1 節 |
| `Linearity.mdl` | 線性系統的齊次性與重疊原理 | 第 2 節 |
| `FreqResp.mdl` | 弦波輸入的穩態響應＝頻率響應 | 第 3 節 |

只用 Colab 內建的 numpy、scipy、plotly。圖上的文字用英文。

In [1]:
import numpy as np
from scipy import signal
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def step(t, t0=1.0, after=1.0, before=0.0):
    """Simulink 的 Step 方塊（預設 Step time = 1、Initial value = 0、Final value = 1）"""
    return after if t >= t0 else before


class TF:
    """Simulink 的 Transfer Fcn 方塊：num、den 是 s 多項式係數（高次在前，同 MATLAB）。
    內部轉成狀態空間 x' = Ax + Bu、y = Cx + Du；n 是狀態數（＝分母階數）。"""
    def __init__(self, num, den):
        A, B, C, D = signal.tf2ss(num, den)
        self.A, self.B, self.C, self.D = A, B[:, 0], C[0], D[0, 0]
        self.n = len(A)

    def dx(self, x, u):
        return self.A @ x + self.B * u

    def y(self, x, u=0.0):
        return self.C @ x + self.D * u


def simulate(rhs, x0, T, breaks=(), dt=0.01, max_step=0.05):
    """仿 Simulink 從 t = 0 跑到 T。breaks：輸入跳變的時刻（例如 Step），在這些點分段積分，
    跳變前那一段一律用跳變前的輸入值。回傳等間隔 dt 的 t 與狀態陣列 X（X[i] 是第 i 個狀態的時間序列）。"""
    n = round(T / dt)
    t = np.linspace(0, T, n + 1)
    cuts = [0.0] + sorted(float(b) for b in breaks if 0 < b < T) + [float(T)]
    x = np.array(x0, dtype=float)
    X = np.empty((len(x), n + 1))
    for a, b in zip(cuts[:-1], cuts[1:]):
        tb = np.nextafter(b, a)
        sol = solve_ivp(lambda s, z: rhs(min(s, tb), z), (a, b), x, method="RK45",
                        rtol=1e-9, atol=1e-12, max_step=max_step, dense_output=True)
        i0, i1 = round(a / dt), round(b / dt)
        idx = slice(i0, i1 + 1 if b == T else i1)
        X[:, idx] = sol.sol(t[idx])
        x = sol.y[:, -1]
    return t, X


def scope(t, panels, title, h=None, npts=600):
    """像 Simulink 的 Scope：panels = {子圖標題: [(曲線名稱, 數列), …]}。點數太多時等距抽樣到約 npts 點（只影響畫圖）。"""
    k = max(1, int(np.ceil(len(t) / npts)))
    fig = make_subplots(rows=len(panels), cols=1, shared_xaxes=True, subplot_titles=list(panels), vertical_spacing=0.08)
    for i, traces in enumerate(panels.values(), 1):
        for name, y in traces:
            fig.add_scatter(x=np.round(t[::k], 3).tolist(), y=np.round(np.asarray(y)[::k], 4).tolist(),
                            name=name, mode="lines", row=i, col=1)
    fig.update_xaxes(title_text="t (s)", row=len(panels), col=1)
    fig.update_layout(title=title, height=h or 260 + 220 * len(panels), template="plotly_white",
                      margin=dict(l=60, r=20, t=70, b=50))
    fig.show()

## 1. 同一個系統的三種 Simulink 模型 `TfSim.mdl`、`sssim.mdl`、`IntgratorM.mdl`

系統：$\ddot x+5\dot x+6x=f(t)=10\sin t$，$x(0)=\dot x(0)=0$；三個模型都是 Sine Wave（振幅 10、1 rad/s）輸入、模擬 10 秒。

* `TfSim.mdl`：Transfer Fcn 方塊 $T(s)=\dfrac{1}{s^2+5s+6}$。
* `sssim.mdl`：State-Space 方塊 $A=\begin{bmatrix}0&1\\-6&-5\end{bmatrix}$、$B=\begin{bmatrix}0\\1\end{bmatrix}$、$C=\begin{bmatrix}1&0\end{bmatrix}$、$D=0$。
* `IntgratorM.mdl`：兩個積分器串接，$\ddot x=f(t)-5\dot x-6x$（Sum 方塊符號 `-+-`：$-5\dot x$、$+f$、$-6x$）。

網頁上的解析解（MATLAB `dsolve`）：$x=\underbrace{\sin t-\cos t}_{\text{穩態}}\ \underbrace{-e^{-3t}+2e^{-2t}}_{\text{暫態}}$。

In [2]:
f = lambda t: 10 * np.sin(t)                                          # Sine Wave：Amplitude 10、Frequency 1 rad/s

# TfSim.mdl
G = TF([1], [1, 5, 6])
t, X = simulate(lambda t, x: G.dx(x, f(t)), np.zeros(G.n), 10)
x_tf = G.C @ X

# sssim.mdl
A = np.array([[0, 1], [-6, -5]]); B = np.array([0, 1]); C = np.array([1, 0])
t, Y = simulate(lambda t, y: A @ y + B * f(t), [0, 0], 10)
x_ss = C @ Y


# IntgratorM.mdl：Integrator1 輸出 = x'，Integrator 輸出 = x
def rhs(t, s):
    x, v = s
    acc = -5 * v + f(t) - 6 * x                                       # Sum 方塊 "-+-"（Gain = 5、Gain1 = 6）
    return [v, acc]


t, Z = simulate(rhs, [0, 0], 10)
x_int = Z[0]

x_exact = np.sin(t) - np.cos(t) - np.exp(-3 * t) + 2 * np.exp(-2 * t)
for name, xx in [("TfSim", x_tf), ("sssim", x_ss), ("IntgratorM", x_int)]:
    print(f"{name:11s}: max |x - analytic| = {np.abs(xx - x_exact).max():.1e}")
scope(t, {"Scope (all three models give the same x(t))": [("TfSim", x_tf), ("sssim", x_ss), ("IntgratorM", x_int),
                                                         ("analytic", x_exact)]},
      "x'' + 5x' + 6x = 10 sin t")

TfSim      : max |x - analytic| = 4.5e-10
sssim      : max |x - analytic| = 4.5e-10
IntgratorM : max |x - analytic| = 4.5e-10


三種模型的結果和解析解一致（差異只在數值誤差）。網頁上的提醒：轉移函數只描述輸入輸出關係，
狀態空間與積分器模型則保留了系統內部的狀態（$x$、$\dot x$），可以直接拿出來看或回授。

## 2. 線性系統的模擬 `Linearity.mdl`

五個相同的 $G(s)=\dfrac{1}{s^2+0.2s+1}$，模擬 10 秒：

* **齊次性**：Step（$t=1$ 跳到 1）→ $G$ → $Y_1$；Step → Gain 2 → $G$ → $Y_2$。Scope「2Y1-Y2」應為 0。
* **重疊原理**：$u_1=\sin t$ → $G$ → $y_1$；$u_2$＝Step → $G$ → $y_2$；$u_1+u_2$ → $G$ → $y_3$。Scope「y1+y2-y3」應為 0。

In [3]:
G = TF([1], [1, 0.2, 1])
n = G.n


def rhs(t, s):
    x1, x2, x3, x4, x5 = (s[i * n:(i + 1) * n] for i in range(5))
    u, u1, u2 = step(t), np.sin(t), step(t)
    return np.r_[G.dx(x1, u), G.dx(x2, 2 * u),                         # 齊次性
                 G.dx(x3, u1), G.dx(x4, u2), G.dx(x5, u1 + u2)]        # 重疊原理


t, X = simulate(rhs, np.zeros(5 * n), 10, breaks=(1,))
Y1, Y2, y1, y2, y3 = (G.C @ X[i * n:(i + 1) * n] for i in range(5))
print(f"homogeneity : max |2*Y1 - Y2|     = {np.abs(2 * Y1 - Y2).max():.1e}")
print(f"superposition: max |y1 + y2 - y3| = {np.abs(y1 + y2 - y3).max():.1e}")
scope(t, {"Y1, Y2 (step, 2 x step)": [("Y1", Y1), ("Y2", Y2), ("2Y1-Y2", 2 * Y1 - Y2)],
          "y1 (sin t), y2 (step), y3 (sin t + step)": [("y1", y1), ("y2", y2), ("y3", y3), ("y1+y2-y3", y1 + y2 - y3)]},
      "Linearity.mdl")

homogeneity : max |2*Y1 - Y2|     = 0.0e+00
superposition: max |y1 + y2 - y3| = 2.7e-15


兩個 Scope 都是 0：輸入放大 2 倍，輸出也放大 2 倍；兩個輸入相加，輸出就是各自輸出的和。
網頁上的反例 $y=au+b$（$b\ne0$）就不滿足：$u_1+u_2$ 的輸出比 $y_1+y_2$ 少了一個 $b$。

## 3. 頻率響應 `FreqResp.mdl`

$\ddot y+0.2\dot y+y=\sin(\omega t)$，$G(s)=\dfrac{1}{s^2+0.2s+1}$；模型用 Sine Wave（1 rad/s）輸入、模擬 70 秒，
以 To Workspace 存下 `InP`、`OutP`、`t`，再 `plot(t, InP, t, OutP)`。

頻率響應的定義：穩定線性系統受弦波 $A\sin\omega t$ 輸入，穩態輸出為 $B\sin(\omega t+\phi)$，
$B=A|G(j\omega)|$、$\phi=\angle G(j\omega)$。$\omega=1$ 正好是 $G$ 的共振頻率：$|G(j1)|=\dfrac{1}{0.2}=5$、$\angle G(j1)=-90^\circ$。

In [4]:
def sine_test(w, T=70):
    """FreqResp.mdl：Sine Wave（頻率 w）→ G → 存 InP、OutP；用最後兩個週期擬合 OutP = B sin(wt + phi)"""
    tt, X = simulate(lambda t, x: G.dx(x, np.sin(w * t)), np.zeros(G.n), T)
    InP, OutP = np.sin(w * tt), G.C @ X
    m = tt >= T - 2 * (2 * np.pi / w)
    a, b = np.linalg.lstsq(np.c_[np.sin(w * tt[m]), np.cos(w * tt[m])], OutP[m], rcond=None)[0]
    return tt, InP, OutP, np.hypot(a, b), np.degrees(np.arctan2(b, a))


t, InP, OutP, B, phi = sine_test(1.0)
Gj = 1 / ((1j) ** 2 + 0.2 * 1j + 1)
print(f"w = 1: steady state B = {B:.2f}, phi = {phi:.1f} deg;  |G(j1)| = {abs(Gj):.2f}, angle = {np.degrees(np.angle(Gj)):.1f} deg")
scope(t, {"plot(t, InP, t, OutP)": [("InP = sin t", InP), ("OutP", OutP)]}, "FreqResp.mdl: 1/(s^2+0.2s+1), input sin t")

for w in (0.5, 2.0):
    _, _, _, B, phi = sine_test(w)
    Gj = 1 / ((1j * w) ** 2 + 0.2 * 1j * w + 1)
    print(f"w = {w}: steady state B = {B:.3f}, phi = {phi:.1f} deg;  |G(jw)| = {abs(Gj):.3f}, angle = {np.degrees(np.angle(Gj)):.1f} deg")

w = 1: steady state B = 4.99, phi = -90.0 deg;  |G(j1)| = 5.00, angle = -90.0 deg


w = 0.5: steady state B = 1.322, phi = -7.6 deg;  |G(jw)| = 1.322, angle = -7.6 deg
w = 2.0: steady state B = 0.330, phi = -172.4 deg;  |G(jw)| = 0.330, angle = -172.4 deg


模擬的穩態振幅與相位和 $|G(j\omega)|$、$\angle G(j\omega)$ 一致（$\omega=1$ 時阻尼小，暫態以 $e^{-0.1t}$ 衰減，
到 70 秒仍殘留約 0.01，所以擬合出 4.99 而不是 5.00）。把所有頻率的 $|G(j\omega)|$、$\angle G(j\omega)$ 畫出來就是 Bode 圖
（網頁用 `ltiview(sys)` 選 Bode），下面標出上面試的三個頻率。

In [5]:
w = np.logspace(-1, 1, 400)
Gw = 1 / ((1j * w) ** 2 + 0.2 * 1j * w + 1)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08)
fig.add_scatter(x=w, y=np.round(20 * np.log10(np.abs(Gw)), 4), name="|G(jw)| (dB)", row=1, col=1)
fig.add_scatter(x=w, y=np.round(np.degrees(np.angle(Gw)), 3), name="angle G(jw) (deg)", row=2, col=1)
for wk in (0.5, 1.0, 2.0):
    g = 1 / ((1j * wk) ** 2 + 0.2 * 1j * wk + 1)
    fig.add_scatter(x=[wk], y=[20 * np.log10(abs(g))], mode="markers", marker=dict(size=9, color="#d62728"), showlegend=False, row=1, col=1)
    fig.add_scatter(x=[wk], y=[np.degrees(np.angle(g))], mode="markers", marker=dict(size=9, color="#d62728"), showlegend=False, row=2, col=1)
fig.update_xaxes(type="log", title_text="w (rad/s)", row=2, col=1)
fig.update_yaxes(title_text="dB", row=1, col=1)
fig.update_yaxes(title_text="deg", row=2, col=1)
fig.update_layout(title="Bode plot of 1/(s^2+0.2s+1)", height=560, template="plotly_white")
fig.show()